In [1]:
import pypsa
from tz_pypsa.constraints import (constr_max_annual_utilisation_generator, 
                                  constr_min_annual_utilisation_generator,
                                  constr_max_annual_utilisation_links,
                                  constr_min_annual_utilisation_links,
                                  constr_max_annual_utilisation_storage_discharge,
                                  constr_min_annual_utilisation_storage_discharge,
                                  constr_max_annual_utilisation_storage_charge,
                                  constr_min_annual_utilisation_storage_charge,
                                  constr_soc_intraday_profile,
                                  constr_soc_weekly_profile
                                  )

import plotly.express as px
import pandas as pd     
import numpy as np
import xarray as xr
import os
os.environ['GRB_LICENSE_FILE'] = '/home/jy/opt/gurobi/gurobi.lic'

In [2]:
n = pypsa.Network()
n.import_from_netcdf("/home/jy/tz-amp/.amp/client-earth_CE-A-OCCTO-003_v3/platform_network.nc")

INFO:pypsa.io:Imported network platform_network.nc has buses, carriers, generators, links, loads, storage_units


In [3]:
n.generators['carrier'] = n.generators['type']
n.links['carrier'] = n.links['type']
n.storage_units['carrier'] = n.storage_units['type']

In [4]:
n.generators.loc[(n.generators.carrier == 'coal-unspecified'), 'ramp_limit_up'] = 0.4
n.generators.loc[(n.generators.carrier == 'coal-unspecified'), 'ramp_limit_down'] = 0.4

n.generators.loc[(n.generators.carrier == 'gas-unspecified'), 'ramp_limit_up'] = 0.9
n.generators.loc[(n.generators.carrier == 'gas-unspecified'), 'ramp_limit_down'] = 0.9

In [5]:
n.storage_units['standing_loss'] = 5e-05
n.storage_units['cyclic_state_of_charge'] = True

In [ ]:
target_regions = ([
                    "HK", #not fine
                    "TH", #not fine
                    "TK", #not fine
                     "CB", #fine
                     "HR", #fine
                    "KA",  #not fine
                    "CG", #fine
                    "SH",  #not fine
                    "KY" #not fine
                   ]
                   )
region_pattern = '|'.join(target_regions)
region_units = n.storage_units.index[n.storage_units.bus.str.contains(region_pattern)]

time_mask = n.storage_units_t.state_of_charge_set.index.get_level_values('timestep').hour == 21
n.storage_units_t.state_of_charge_set.loc[time_mask, region_units] = np.nan

In [6]:
target_regions = ([
                    "HK", #not fine
                    "TH", #not fine
                    "TK", #not fine
                     "CB", #fine
                     "HR", #fine
                    "KA",  #not fine
                    "CG", #fine
                    "SH",  #not fine
                    "KY" #not fine
                   ]
                   )
region_pattern = '|'.join(target_regions)
region_units = n.storage_units.index[n.storage_units.bus.str.contains(region_pattern)]

time_mask = (n.storage_units_t.state_of_charge_set.index.get_level_values('timestep').hour == 13) | (n.storage_units_t.state_of_charge_set.index.get_level_values('timestep').hour == 5)| (n.storage_units_t.state_of_charge_set.index.get_level_values('timestep').hour == 21)
n.storage_units_t.state_of_charge_set.loc[time_mask, region_units] = np.nan

In [ ]:
target_regions = ([
                    "HK", #not fine
                    "TH", #not fine
                    "TK", #not fine
                    "CB", #fine
                    "HR", #fine
                    "KA",  #not fine
                    "CG", #fine
                    "SH",  #not fine
                    "KY" #not fine
        
])
region_pattern = '|'.join(target_regions)
region_units = n.storage_units.index[n.storage_units.bus.str.contains(region_pattern)]

# Get masks for hour 1 and hour 2
hour_1_mask = n.storage_units_t.state_of_charge_set.index.get_level_values('timestep').hour == 5
hour_2_mask = n.storage_units_t.state_of_charge_set.index.get_level_values('timestep').hour == 13
hour_3_mask = n.storage_units_t.state_of_charge_set.index.get_level_values('timestep').hour == 21

# Get hour 1 values for the target regions
hour_1_values = n.storage_units_t.state_of_charge_set.loc[hour_1_mask, region_units]
hour_2_values = n.storage_units_t.state_of_charge_set.loc[hour_2_mask, region_units]
hour_3_values = n.storage_units_t.state_of_charge_set.loc[hour_3_mask, region_units]
# Multiply by 0.95 and assign to hour 2
# Note: The indices should align automatically since both masks select the same dates
n.storage_units_t.state_of_charge_set.loc[hour_1_mask, region_units] = hour_1_values.values * 0.7
n.storage_units_t.state_of_charge_set.loc[hour_2_mask, region_units] = hour_2_values.values * 0.7
n.storage_units_t.state_of_charge_set.loc[hour_3_mask, region_units] = hour_3_values.values * 0.7

In [ ]:
n.storage_units_t.state_of_charge_set[n.storage_units_t.state_of_charge_set.notna().any(axis=1)]

In [7]:
n.generators.loc[n.generators.carrier == 'nuclear', 'p_min_pu'] = 0.59
n.generators.loc[n.generators.carrier == 'nuclear', 'p_max_pu'] = 0.59

n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_min_pu'] = 0.585
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_max_pu'] = 0.585
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_min_pu'] = 0.585
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_max_pu'] = 0.585
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_min_pu'] = 0.585
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_max_pu'] = 0.585

In [ ]:
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CB'), 'p_max_pu'] = 0.30
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CB'), 'p_min_pu'] = -0.50

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-SH'), 'p_max_pu'] = 0.63
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-SH'), 'p_min_pu'] = -0.70

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HR'), 'p_max_pu'] = 0.50
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HR'), 'p_min_pu'] = -0.70

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KA'), 'p_max_pu'] = 0.50
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KA'), 'p_min_pu'] = -0.65

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CG'), 'p_max_pu'] = 0.50
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CG'), 'p_min_pu'] = -0.70

In [8]:
n.generators_t.p_min_pu = n.generators_t.p_max_pu.filter(regex='biomass|geothermal|hydro')

In [9]:
# p_max_pu
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_max_pu'] = 0.46
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_max_pu'] = 0.57
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_max_pu'] = 0.62
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_max_pu'] = 0.63
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_max_pu'] = 0.60
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_max_pu'] = 0.60
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_max_pu'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_max_pu'] = 0.59
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_max_pu'] = 0.51

# p_min_pu
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_min_pu'] = 0.10
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_min_pu'] = 0.15
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_min_pu'] = 0.20
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_min_pu'] = 0.20
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_min_pu'] = 0.20
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_min_pu'] = 0.20
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_min_pu'] = 0.20
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_min_pu'] = 0.20
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_min_pu'] = 0.20

In [10]:
# max_utilisation_rate
# coal
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.34
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.445
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.55
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.51
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.41
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.45
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.365

# gas
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.10
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.09
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.23

In [11]:
# min_utilisation_rate
# coal
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.34
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.445
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.365

# gas
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.14
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.095
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.16
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.23
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.10
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.225
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.05
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'min_utilisation_rate'] = 0.085
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.04

In [12]:
# max_utilisation_rate - transmission
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HK~GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.48
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TH~GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TH~GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.5875
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TK~GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TK~GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.44
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HR~GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.33
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HR~GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.18
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.09
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.25
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.40
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-SH~GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KY~GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.34
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KY~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.83

In [13]:
# min_utilisation_rate - transmission
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HK~GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.48
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TH~GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.5875
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.25
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.04
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.09
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.19
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.19
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-SH~GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.91

In [14]:
# max_utilisation_rate
# hydro-pumped-storage
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HK'), 'discharge_min_utilisation_rate'] = 0.126
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HK'), 'charge_max_utilisation_rate'] = 0.180

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TH'), 'discharge_min_utilisation_rate'] = 0.136
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TH'), 'charge_max_utilisation_rate'] = 0.195

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TK'), 'discharge_min_utilisation_rate'] = 0.121
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TK'), 'charge_max_utilisation_rate'] = 0.174

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CB'), 'discharge_min_utilisation_rate'] = 0.059
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CB'), 'charge_max_utilisation_rate'] = 0.085

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HR'), 'discharge_min_utilisation_rate'] = 0.055
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HR'), 'charge_max_utilisation_rate'] = 0.080

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KA'), 'discharge_min_utilisation_rate'] = 0.056
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KA'), 'charge_max_utilisation_rate'] = 0.080

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CG'), 'discharge_min_utilisation_rate'] = 0.056
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CG'), 'charge_max_utilisation_rate'] = 0.081

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-SH'), 'discharge_min_utilisation_rate'] = 0.064
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-SH'), 'charge_max_utilisation_rate'] = 0.092

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KY'), 'discharge_max_utilisation_rate'] = 0.057
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KY'), 'charge_max_utilisation_rate'] = 0.083

In [15]:
n.optimize.create_model()

constr_max_annual_utilisation_generator(n, carriers='coal-unspecified|gas-unspecified') # Set max annual utilisation for these generators
constr_min_annual_utilisation_generator(n, carriers='coal-unspecified|gas-unspecified') # Set min annual utilisation for these generators
constr_max_annual_utilisation_links(n, carriers='transmission') # Set max annual utilisation for these links
constr_min_annual_utilisation_links(n, carriers='transmission') # Set min annual utilisation for these links
constr_max_annual_utilisation_storage_discharge(n, carriers='hydro-pumped-storage-unspecified') # Set max annual utilisation for these storage units
constr_min_annual_utilisation_storage_discharge(n, carriers='hydro-pumped-storage-unspecified') # Set min annual utilisation for these storage units
constr_max_annual_utilisation_storage_charge(n, carriers='hydro-pumped-storage-unspecified') # Set min annual utilisation for these storage units
constr_soc_intraday_profile(
    n, 
    max_csv="/home/jy/tz-amp/.amp/client-earth_CE-A-OCCTO-003_testing/templates/state_of_charge_intraday_profile_annual_max.csv",
    min_csv="/home/jy/tz-amp/.amp/client-earth_CE-A-OCCTO-003_testing/templates/state_of_charge_intraday_profile_annual_min.csv"
)
constr_soc_weekly_profile(
    n, 
    max_csv="/home/jy/tz-amp/.amp/client-earth_CE-A-OCCTO-003_testing/templates/state_of_charge_weekly_profile_annual_max.csv",
    min_csv="/home/jy/tz-amp/.amp/client-earth_CE-A-OCCTO-003_testing/templates/state_of_charge_weekly_profile_annual_min.csv",
    day_shift=2
)

Index(['hydro-pumped-storage-unspecified:GRIDREGION-JPN-SH',
       'hydro-pumped-storage-unspecified:GRIDREGION-JPN-HR',
       'hydro-pumped-storage-unspecified:GRIDREGION-JPN-CB',
       'hydro-pumped-storage-unspecified:GRIDREGION-JPN-HK',
       'hydro-pumped-storage-unspecified:GRIDREGION-JPN-KA',
       'hydro-pumped-storage-unspecified:GRIDREGION-JPN-CG',
       'hydro-pumped-storage-unspecified:GRIDREGION-JPN-TK',
       'hydro-pumped-storage-unspecified:GRIDREGION-JPN-TH',
       'hydro-pumped-storage-unspecified:GRIDREGION-JPN-KY'],
      dtype='object', name='StorageUnit')
Index(['transmission:GRIDREGION-JPN-KY~GRIDREGION-JPN-SH',
       'transmission:GRIDREGION-JPN-SH~GRIDREGION-JPN-KY',
       'transmission:GRIDREGION-JPN-KY~GRIDREGION-JPN-CG',
       'transmission:GRIDREGION-JPN-HK~GRIDREGION-JPN-TH',
       'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CG',
       'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-HR',
       'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-

['coal-unspecified:GRIDREGION-JPN-SH', 'coal-unspecified:GRIDREGION-JPN-HR', 'coal-unspecified:GRIDREGION-JPN-CB', 'coal-unspecified:GRIDREGION-JPN-HK', 'coal-unspecified:GRIDREGION-JPN-KA', 'coal-unspecified:GRIDREGION-JPN-CG', 'coal-unspecified:GRIDREGION-JPN-TK', 'coal-unspecified:GRIDREGION-JPN-TH', 'coal-unspecified:GRIDREGION-JPN-KY', 'gas-unspecified:GRIDREGION-JPN-SH', 'gas-unspecified:GRIDREGION-JPN-HR', 'gas-unspecified:GRIDREGION-JPN-KA']
['coal-unspecified:GRIDREGION-JPN-HK', 'coal-unspecified:GRIDREGION-JPN-TK', 'coal-unspecified:GRIDREGION-JPN-TH', 'coal-unspecified:GRIDREGION-JPN-KY', 'gas-unspecified:GRIDREGION-JPN-SH', 'gas-unspecified:GRIDREGION-JPN-HR', 'gas-unspecified:GRIDREGION-JPN-CB', 'gas-unspecified:GRIDREGION-JPN-HK', 'gas-unspecified:GRIDREGION-JPN-KA', 'gas-unspecified:GRIDREGION-JPN-CG', 'gas-unspecified:GRIDREGION-JPN-TK', 'gas-unspecified:GRIDREGION-JPN-TH', 'gas-unspecified:GRIDREGION-JPN-KY']
['transmission:GRIDREGION-JPN-KY~GRIDREGION-JPN-SH', 'transm

In [16]:
n.optimize.solve_model(
    solver_name='gurobi',
    # solver_options={
    #     'threads': 8,
    #     'method': 2, # barrier
    #     'crossover': 0,
    #     'BarConvTol': 1.e-6,
    #     'Seed': 123,
    #     'AggFill': 0,
    #     'PreDual': 0,
    # },
    # io_api="direct",
    # env=None,
)

INFO:linopy.model: Solve problem using Gurobi solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 5/5 [00:00<00:00,  5.39it/s]
INFO:linopy.io: Writing time: 7.76s


Set parameter WLSAccessID


INFO:gurobipy:Set parameter WLSAccessID


Set parameter WLSSecret


INFO:gurobipy:Set parameter WLSSecret


...

INFO:gurobipy:Solved in 443063 iterations and 49.00 seconds (72.49 work units)


Optimal objective  3.906414239e+10


INFO:gurobipy:Optimal objective  3.906414239e+10


INFO:gurobipy:Warning: environment still referenced so free is deferred (Continue to use WLS)
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 1217640 primals, 2908907 duals
Objective: 3.91e+10
Solver model: available
Solver message: 2

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-fix-p-ramp_limit_up, Generator-fix-p-ramp_limit_down, Link-fix-p-lower, Link-fix-p-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, StorageUnit-state_of_charge_set, StorageUnit-energy_balance were not assigned to the network.


('ok', 'optimal')

In [ ]:
n.storage_units[['p_max_pu', 'p_min_pu']]

In [ ]:
n.storage_units_t.p_dispatch.sum() / (n.storage_units.p_nom * 8760)

In [ ]:
n.storage_units_t.p_store.sum() / (n.storage_units.p_nom * 8760)


In [ ]:
n.storage_units_t.p_dispatch.sum() / n.storage_units_t.p_store.sum()

In [ ]:
import plotly.express as px

px.line(n.storage_units_t.state_of_charge.filter(regex='GRIDREGION-JPN-HK').reset_index(drop=True, level=0))

In [ ]:
n.links_t.p0.sum() / (n.links.p_nom * 8760)

In [ ]:
n.links_t.p0.sum()

In [ ]:
n.storage_units_t.state_of_charge

In [ ]:
n.export_to_netcdf("/home/jy/tz-amp/.amp/client-earth_CE-A-OCCTO-003_testing-v5/platform_network.solved.nc")